Class implementation for CAX simulation module

```26/05/2026```

In [3]:
import Shadow
import numpy as np
import matplotlib.pyplot as plt

What do I need to accomplish? I need to be able to simulate the passage of light through the CAX beamline, from source through slits to DVF screen. I then need to be able to extract the resulting image at a given $y$ position (optical axis). Then I need to perform movements to the mirror and regenerate the image.

To create a general `CAXSim` class, I'll first write a basic `OpticalElement` wrapper class for Shadow OEs, then I'll create children with more specific methods, all the way to a `BeamLine` class that possesses the necessary attributes and methods to perform simulations and modify the system. 

In [ ]:
class OpticalElement:
    """
    A wrapper class for a Shadow optical element, 
    providing methods to load specifications and interact with the element.
    """

    def __init__(self, name: str, specification_file: str):
        self.name = name
        self.shadow_oe = Shadow.OE()
        self.specification_file = specification_file
        self.load_specification()

        # General attributes
        self.unit = self.shadow_oe.unit()
        
    @property
    def orientation(self):
        """
        Optical element orientation defined as the relative angle of the normal
        of the OE surface with respect to the previous element's normal, measured
        from the previous element looking towards the beam's direction of propagation.
        Result in degrees, positive means counter-clockwise.
        """
        return self.shadow_oe.ALPHA
    
    def load_specification(self, specification_file: str = None):
        """
        Load the optical element specification from a file.
        
        Parameters:
            specification_file (str): The path to the specification file. 
            If None, uses the default specification file.
        """

        if specification_file is None:
            specification_file = self.specification_file
        self.shadow_oe.load(specification_file)

class Source():
    """
    A wrapper class for a Shadow source optical element.
    """

    def __init__(self, name: str, specification_file: str):
        self.name = name
        self.shadow_oe = Shadow.Source()
        self.specification_file = specification_file
        self.load_specification()
    
    def load_specification(self, specification_file: str = None):
        """
        Load the optical element specification from a file.
        
        Parameters:
            specification_file (str): The path to the specification file. 
            If None, uses the default specification file.
        """

        if specification_file is None:
            specification_file = self.specification_file
        self.shadow_oe.load(specification_file)

class BendingMagnet(Source):
    """
    A wrapper class for a Shadow bending magnet optical element.
    """

    def __init__(self, name: str, specification_file: str):
        super().__init__(name, specification_file)
        
        if self.shadow_oe.FDISTR != 4:
            raise ValueError(f"File {specification_file} is not"
                             f" configured for a synchrotron source.")

class ToroidalMirror(OpticalElement):
    """
    A wrapper class for a Shadow toroidal mirror optical element.
    """

    def __init__(self, name: str, specification_file: str):
        super().__init__(name, specification_file)
        
        if self.shadow_oe.FMIRR != 3:
            raise ValueError(f"File {specification_file} is not"
                             f" configured for a toroidal mirror: "
                             f"FMIRR = {self.shadow_oe.FMIRR}")


In [7]:
source = BendingMagnet("B1", "source.props")

In [13]:
mirror = ToroidalMirror("M1", "mirror.props")
mirror.orientation

90.0